# 2D Gaussian Splatting — DTU Pipeline

논문(2DGS, Huang et al. SIGGRAPH 2024)의 DTU 벤치마크를 Colab에서 재현하는 파이프라인.

**데이터**: DTU MVS scan — **저자가 직접 제공**하는 전처리본 사용 (scan당 ~49~64 뷰, 1600×1200, 학습 시 `-r 2` → 800×600)

**데이터 출처** (README.md 참조):
| 데이터 | 용도 | 제공 주체 | 크기 |
|---|---|---|---|
| [DTU+COLMAP preprocessed](https://drive.google.com/drive/folders/1SJFgt8qhQomHX55Q4xSvYE2C6-8tFll9) | 학습 | **2DGS 저자 공식 Google Drive** | 3.5 GB |
| [Hugging Face mirror](https://huggingface.co/datasets/dylanebert/2DGS) | 학습 (대안) | 커뮤니티 미러 | 동일 |
| [DTU Official GT](https://roboimagedata.compute.dtu.dk/?page_id=36) | Chamfer 평가 | DTU 본가 (저자 아님) | ~5 GB |

**하이퍼파라미터(`scripts/dtu_eval.py` 기준)**:
- 학습: `--depth_ratio 1.0 -r 2 --lambda_dist 1000`
- 렌더링/메쉬: `--depth_ratio 1.0 --num_cluster 1 --voxel_size 0.004 --sdf_trunc 0.016 --depth_trunc 3.0`
- `--eval`은 사용하지 않음 (DTU는 PSNR이 아니라 mesh Chamfer로 평가하므로 전체 뷰를 학습에 사용)

**평가**: DTU 공식 GT point cloud와 Chamfer distance 비교 (`scripts/eval_dtu/evaluate_single_scene.py`)

**저자 전처리본 특징** (README:125):
- mask가 이미지의 **alpha 채널**에 저장됨
- 저자가 `scene/cameras.py:43-48`의 알파 블렌딩 라인을 주석 처리해둠 (이 레포에 이미 적용됨) → 배경 포함 학습 가능

**미쓰비시 파이프라인과의 차이**:
1. 데이터가 이미 COLMAP 포맷 → calib ini 변환 불필요
2. 배경 전처리 불필요 (alpha 채널 + 저자 코드 수정으로 처리됨)
3. GT가 point cloud이므로 depth MAE 대신 **Chamfer distance** 사용
4. `lambda_dist=1000` (논문 DTU 세팅, 미쓰비시의 0.1보다 4자리수 큼 — 실물 스케일 + 정밀 geometry 요구)

## 1. Environment Setup

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
# Google Drive 마운트 (데이터/체크포인트 영속 저장용)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE = "/content/drive/MyDrive/2dgs_results_dtu"
import os
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f"Drive mounted. Save path: {DRIVE_SAVE}")

In [ ]:
import os
ROOT = "/content"
REPO = os.path.join(ROOT, "2d-gaussian-splatting")

# DTU scan 선택 (논문 평가 15개 중 1개를 기본값으로)
# scan24=bear, scan37=?, scan40=?, scan55=owl, scan63=dog, scan65=?,
# scan69=?, scan83=?, scan97=?, scan105=toy, scan106=?, scan110=?,
# scan114=?, scan118=?, scan122=?
SCAN_ID = 24
SCENE = f"scan{SCAN_ID}"

DTU_ROOT = os.path.join(ROOT, "dtu")          # NeuS 전처리본 루트 (여러 scan 담김)
DATA = os.path.join(DTU_ROOT, SCENE)          # 개별 scan 경로 (train.py -s 대상)
OUTPUT = os.path.join(ROOT, "output_dtu", SCENE)
DTU_GT = os.path.join(ROOT, "dtu_gt")         # 공식 GT point cloud 루트

print(f"Scene: {SCENE}")
print(f"Data path: {DATA}")
print(f"Output path: {OUTPUT}")

In [ ]:
# 레포 클론 (본인 fork — 미쓰비시 파이프라인과 동일 코드베이스 사용)
import os, subprocess
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "--recursive",
         "https://github.com/BAEJUNHAK/2d-gaussian-splatting.git", REPO],
        check=True,
    )
    print(f"Cloned → {REPO}")
else:
    print(f"Already cloned: {REPO}")


In [ ]:
# PyTorch 버전 확인 (Colab 기본 설치된 것 사용)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# ============================================================
# CUDA 확장 빌드 + site-packages 직접 배치
# → train.py subprocess 가 import 할 수 있도록 확정
# ============================================================
import os, sys, site, subprocess, glob, shutil, importlib, torch

# [1] 의존성 (한 줄로 — 백슬래시 continuation 은 Colab 에서 가끔 깨짐)
!pip install -q plyfile scikit-image==0.21.0 lpips==0.1.4 trimesh==4.3.2 open3d mediapy==1.1.2 opencv-python tqdm pillow tensorboard scipy
!pip install -q ninja wheel setuptools pybind11

# [2] CUDA 환경
nvcc = subprocess.getoutput("which nvcc").strip()
assert nvcc, "nvcc 없음"
os.environ["CUDA_HOME"] = os.path.dirname(os.path.dirname(nvcc))
os.environ["PATH"] = f"{os.path.dirname(nvcc)}:" + os.environ.get("PATH", "")
print(f"CUDA_HOME = {os.environ['CUDA_HOME']}, nvcc = {nvcc}")
print(f"torch = {torch.__version__} (cuda {torch.version.cuda})")

# [3] simple_knn FLT_MAX 패치
knn_cu = os.path.join(REPO, "submodules", "simple-knn", "simple_knn.cu")
with open(knn_cu) as f:
    src = f.read()
if "#include <cfloat>" not in src:
    with open(knn_cu, "w") as f:
        f.write(src.replace('#include "simple_knn.h"',
                            '#include "simple_knn.h"\n#include <cfloat>'))
    print("Patched simple_knn.cu")

# [4] 이전 찌꺼기 완전 제거
SUBMODS = {
    "diff_surfel_rasterization": os.path.join(REPO, "submodules", "diff-surfel-rasterization"),
    "simple_knn":                 os.path.join(REPO, "submodules", "simple-knn"),
}
site_pkg = site.getsitepackages()[0]

for mod_name, d in SUBMODS.items():
    for sub in ["build", f"{mod_name}.egg-info"]:
        p = os.path.join(d, sub)
        if os.path.exists(p):
            shutil.rmtree(p)
    for so in glob.glob(os.path.join(d, "**", "*.so"), recursive=True):
        os.remove(so)

stale_pth = os.path.join(site_pkg, "2dgs_submodules.pth")
if os.path.exists(stale_pth):
    os.remove(stale_pth)
for p in glob.glob(os.path.join(site_pkg, "*surfel*")) + glob.glob(os.path.join(site_pkg, "*simple_knn*")):
    if os.path.isdir(p):
        shutil.rmtree(p)
    else:
        os.remove(p)

# [5] build_ext --inplace
for mod_name, d in SUBMODS.items():
    print(f"\n{'='*60}\nBuilding {mod_name}\n{'='*60}")
    r = subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=d, capture_output=True, text=True, env={**os.environ},
    )
    if r.returncode != 0:
        print("STDOUT:", r.stdout[-1500:])
        print("STDERR:", r.stderr[-2000:])
        raise RuntimeError(f"{mod_name} 빌드 실패")
    so_files = glob.glob(os.path.join(d, mod_name, "_C*.so"))
    print(f"✓ built .so: {so_files}")

# [6] site-packages 에 패키지 통째 복사
for mod_name, d in SUBMODS.items():
    src_dir = os.path.join(d, mod_name)
    dst_dir = os.path.join(site_pkg, mod_name)
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    print(f"✓ copied: {src_dir} → {dst_dir}")
    for f in sorted(os.listdir(dst_dir)):
        size = os.path.getsize(os.path.join(dst_dir, f))
        print(f"    {f}  ({size:,} B)")

# [7] subprocess import 테스트 (train.py 와 동일 조건)
print("\n" + "="*60)
print("[7] subprocess import 테스트")
print("="*60)
test_code = (
    "import torch; "
    "import diff_surfel_rasterization; "
    "from simple_knn._C import distCUDA2; "
    "x = torch.rand(100, 3).cuda(); "
    "print('distCUDA2 OK, shape=', distCUDA2(x).shape); "
    "print('diff_surfel_rasterization:', diff_surfel_rasterization.__file__)"
)
test = subprocess.run(
    [sys.executable, "-c", test_code],
    capture_output=True, text=True,
)
print("STDOUT:", test.stdout)
if test.returncode != 0:
    print("STDERR:", test.stderr)
    import ctypes
    for mod, d in SUBMODS.items():
        pkg_dir = os.path.join(site_pkg, mod)
        if not os.path.isdir(pkg_dir):
            continue
        so_name = next((f for f in os.listdir(pkg_dir) if f.startswith("_C") and f.endswith(".so")), None)
        if not so_name:
            continue
        so = os.path.join(pkg_dir, so_name)
        print(f"\nldd {so}:")
        print(subprocess.getoutput(f"ldd {so} | grep -E 'not found|=>' | head -10"))
        try:
            ctypes.CDLL(so, ctypes.RTLD_GLOBAL)
            print(f"  dlopen RTLD_GLOBAL: OK")
        except OSError as e:
            print(f"  dlopen FAIL: {e}")
    raise RuntimeError("subprocess import 실패")
else:
    print("\n✅ subprocess import 성공 → train.py 도 작동")


## 2. DTU Data Download

**데이터 2종이 필요**:

| 데이터 | 용도 | 크기 |
|---|---|---|
| NeuS 전처리본 (이미지 + COLMAP sparse) | 학습 | scan당 ~300~500 MB, 15개 전부 ~6 GB |
| DTU 공식 GT point cloud (SampleSet + Points) | Chamfer 평가 | ~5 GB |

**주의**:
- NeuS 전처리본은 Dropbox/Google Drive로 제공 (2DGS 저자가 사용한 버전은 NeuS repo 링크를 따름)
- 공식 GT는 [roboimagedata.compute.dtu.dk](https://roboimagedata.compute.dtu.dk/?page_id=36)에서 제공 (학술 목적 무료)
- Colab 디스크 용량(~100GB) 안에서 여유 있지만, Drive 영구저장용으로 아카이브 권장

아래 셀은 **scan24 한 개만** 받는 예시. 전체 15개가 필요하면 loop로 교체.

In [ ]:
# ============================================================
# DTU 데이터 다운로드 (로컬 실험 검증 완료)
#
# ▶ 저자 Drive 폴더의 진짜 구조 (embeddedfolderview 로 익명 조회):
#     1SJFgt8qhQomHX55Q4xSvYE2C6-8tFll9/
#     ├── eval_dtu/                          (eval 스크립트 4개, 이미 레포에 포함돼 있음)
#     ├── dtu.tar.gz                         (file ID 1ODiOu72tAGPTnhVn0cFZ9MvymDgcoHxQ)
#     │                                       ← 15개 scan 전부 담긴 단일 tar (3.5GB)
#     └── dtu_results.zip                    (precomputed 결과, 불필요)
#
# ▶ 왜 이게 되는가:
#   - 단일 public 파일 → gdown folder 제한 무관, OAuth 불필요
#   - HTTP 303 → drive.usercontent.google.com 리다이렉트 확인됨 (익명 접근 OK)
#   - gdown 이 virus-scan confirm 토큰 자동 처리
#
# ▶ 이전 시도가 실패한 이유 (기록):
#   - gdown.download_folder: 폴더 구조를 가정했으나 실제론 tar 단일 파일
#   - HuggingFace dylanebert/2DGS: 존재하지 않음 (404)
#   - rclone: drive 백엔드는 공개 파일도 OAuth 토큰 필수
# ============================================================
import os

os.makedirs(DTU_ROOT, exist_ok=True)
DTU_TAR_ID = "1ODiOu72tAGPTnhVn0cFZ9MvymDgcoHxQ"
tar_path   = "/content/dtu.tar.gz"

# 1) gdown 단일 파일 다운로드 (OAuth 불필요, 공개 파일)
if not os.path.exists(DATA):
    !pip install -q gdown
    import gdown

    if not os.path.exists(tar_path):
        print(f"Downloading dtu.tar.gz (~3.5GB, 5~10분)...")
        gdown.download(id=DTU_TAR_ID, output=tar_path, quiet=False)
    else:
        print(f"{tar_path} 이미 존재")

    # 2) 압축 해제
    print(f"\nExtracting to {DTU_ROOT}...")
    !tar xzf {tar_path} -C {DTU_ROOT}
    print("압축 해제 완료")

    # 3) 중첩 디렉토리 평탄화 (tar 가 DTU/scan24 구조면 DTU_ROOT/scan24 로)
    nested = os.path.join(DTU_ROOT, "DTU")
    if os.path.isdir(nested) and not os.path.exists(DATA):
        import shutil
        for item in os.listdir(nested):
            shutil.move(os.path.join(nested, item), os.path.join(DTU_ROOT, item))
        os.rmdir(nested)
        print("Flattened DTU/scan* → scan*")
else:
    print(f"✅ 이미 존재: {DATA}")

# 4) 결과 검증 — 저자 전처리본 구조 (images/, mask/, sparse/, cameras.npz)
if os.path.exists(DATA):
    print(f"\n✅ {DATA}")
    !ls {DATA}
    print("\n구조 체크:")
    for req in ["images", "mask", "sparse", "cameras.npz"]:
        path = os.path.join(DATA, req)
        print(f"   {'✅' if os.path.exists(path) else '❌'} {req}")

    # 공간 확보용: 압축 파일 삭제 (선택)
    # os.remove(tar_path); print("tar.gz 삭제")
else:
    print(f"❌ {DATA} 생성 실패")
    !ls {DTU_ROOT}


In [ ]:
# ============================================================
# 옵션 B: NeuS repo의 DTU 링크 (단일 scan 다운로드, Dropbox)
#   - https://www.dropbox.com/sh/w0y8bbdmxzik3uk/AAAaZffBiJevxQzRskoOYcyja?dl=0
#   - 아래처럼 `dl=1`로 바꾸면 직접 다운로드 가능 (단, 폴더 전체는 zip으로만 가능)
# ============================================================
#
# Dropbox 폴더 전체를 받으려면 수동으로 zip 받은 뒤 Drive에 업로드하는 게 가장 안정적.
# 아래는 예시 — 실제 링크는 수동 확인 필요.

# NEUS_DTU_URL = "https://www.dropbox.com/sh/w0y8bbdmxzik3uk/AAAaZffBiJevxQzRskoOYcyja?dl=1"
# !wget -O {ROOT}/neus_dtu.zip "{NEUS_DTU_URL}"
# !unzip -q {ROOT}/neus_dtu.zip -d {DTU_ROOT}

print("옵션 B는 수동 실행 필요 (위 주석 해제).")

In [ ]:
# ============================================================
# 옵션 C: Google Drive에 미리 업로드해둔 scan24만 복사 (가장 빠름)
# ============================================================
#
# Drive에 DTU/scan24 폴더를 미리 올려둔 경우:

DRIVE_DTU_PATH = f"/content/drive/MyDrive/DTU/{SCENE}"  # 본인 Drive 경로로 교체

if os.path.exists(DRIVE_DTU_PATH) and not os.path.exists(DATA):
    import shutil
    shutil.copytree(DRIVE_DTU_PATH, DATA)
    print(f"Copied {SCENE} from Drive")
elif os.path.exists(DATA):
    print(f"{SCENE} already exists at {DATA}")
else:
    print(f"⚠️  {DRIVE_DTU_PATH} not found. 옵션 A/B를 사용하거나 Drive에 업로드하세요.")

In [ ]:
# 저자가 기대하는 scan 디렉토리 구조 검증
# evaluate_single_scene.py:22-41 에서 요구하는 파일들:
#   - images/*.png          (학습 이미지; RGBA면 alpha 채널이 마스크)
#   - mask/*.png            (evaluate_single_scene.py:40, Chamfer 평가용 컬링 마스크)
#   - cameras.npz           (evaluate_single_scene.py:25-28, scale_mat_i / world_mat_i 키)
#   - sparse/0/*.bin        (train.py 가 요구하는 COLMAP)
import numpy as np

def verify_dtu_scan(scan_dir):
    checks = {}
    # 1. COLMAP sparse
    sparse = None
    for cand in [os.path.join(scan_dir, "sparse", "0"), os.path.join(scan_dir, "sparse")]:
        if os.path.exists(os.path.join(cand, "cameras.bin")) or os.path.exists(os.path.join(cand, "cameras.txt")):
            sparse = cand
            break
    checks['sparse'] = sparse

    # 2. images/
    img_dir = os.path.join(scan_dir, "images")
    checks['images'] = sorted(os.listdir(img_dir)) if os.path.isdir(img_dir) else None

    # 3. mask/  (evaluate_single_scene.py:40 이 여기만 봄 — alpha 채널과는 별도)
    mask_dir_local = os.path.join(scan_dir, "mask")
    checks['mask'] = sorted(os.listdir(mask_dir_local)) if os.path.isdir(mask_dir_local) else None

    # 4. cameras.npz  (evaluate_single_scene.py:25-28)
    npz_path = os.path.join(scan_dir, "cameras.npz")
    if os.path.exists(npz_path):
        cam_npz = np.load(npz_path)
        keys = list(cam_npz.keys())
        n_scale = sum(1 for k in keys if k.startswith("scale_mat_"))
        n_world = sum(1 for k in keys if k.startswith("world_mat_"))
        checks['cameras_npz'] = f"{len(keys)} keys, scale_mat×{n_scale}, world_mat×{n_world}"
    else:
        checks['cameras_npz'] = None

    return checks

if os.path.exists(DATA):
    chk = verify_dtu_scan(DATA)
    print(f"=== {SCENE} 구조 검증 ===")
    for k, v in chk.items():
        if v is None:
            print(f"  ❌ {k}: 없음")
        elif isinstance(v, list):
            print(f"  ✅ {k}: {len(v)} files (예: {v[0]}, {v[-1]})")
        else:
            print(f"  ✅ {k}: {v}")

    missing = [k for k, v in chk.items() if v is None]
    if missing:
        print(f"\n⚠️  누락: {missing}")
        print("   저자 전처리본이 맞는지 확인하세요 (images/, mask/, cameras.npz, sparse/0/ 전부 필요)")
    else:
        if chk['images'] and chk['mask'] and len(chk['images']) != len(chk['mask']):
            print(f"\n⚠️  images({len(chk['images'])}) vs mask({len(chk['mask'])}) 개수 불일치")
        else:
            print(f"\n✅ 모든 구조 OK. 학습+Chamfer 평가 가능.")
else:
    print(f"⚠️  {DATA} 없음 — 먼저 데이터 다운로드 셀을 실행하세요.")

## 3. Data Verification

DTU(NeuS 전처리본)는 이미 COLMAP 포맷으로 제공되므로 변환 불필요.
직접 파서로 카메라/포인트 로드해서 확인만.

In [ ]:
import sys
sys.path.insert(0, REPO)

from scene.colmap_loader import (
    read_intrinsics_binary, read_extrinsics_binary, read_points3D_binary,
    read_intrinsics_text, read_extrinsics_text, read_points3D_text,
    qvec2rotmat,
)

# sparse 경로 자동 탐색
sparse_dir = None
for cand in [os.path.join(DATA, "sparse", "0"), os.path.join(DATA, "sparse")]:
    if os.path.exists(os.path.join(cand, "cameras.bin")) or os.path.exists(os.path.join(cand, "cameras.txt")):
        sparse_dir = cand
        break

assert sparse_dir, f"No COLMAP sparse found in {DATA}"
print(f"Sparse dir: {sparse_dir}")

# binary 우선, fallback to text
try:
    cams = read_intrinsics_binary(os.path.join(sparse_dir, "cameras.bin"))
    imgs = read_extrinsics_binary(os.path.join(sparse_dir, "images.bin"))
    xyz, rgb, err = read_points3D_binary(os.path.join(sparse_dir, "points3D.bin"))
    print("Loaded BINARY COLMAP files")
except Exception:
    cams = read_intrinsics_text(os.path.join(sparse_dir, "cameras.txt"))
    imgs = read_extrinsics_text(os.path.join(sparse_dir, "images.txt"))
    xyz, rgb, err = read_points3D_text(os.path.join(sparse_dir, "points3D.txt"))
    print("Loaded TEXT COLMAP files")

# 카메라
for cid, c in cams.items():
    print(f"Camera {cid}: {c.model} {c.width}x{c.height} params={c.params}")

print(f"\nImages: {len(imgs)}")
print(f"Point cloud: {xyz.shape[0]} points")
print(f"  XYZ range: [{xyz.min(0)}, {xyz.max(0)}]")

In [ ]:
# 포인트/카메라 시각화
import numpy as np
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12, 5))

# 포인트 클라우드
ax1 = fig.add_subplot(121, projection='3d')
subsample = np.random.choice(len(xyz), min(5000, len(xyz)), replace=False)
ax1.scatter(xyz[subsample, 0], xyz[subsample, 1], xyz[subsample, 2],
            c=rgb[subsample] / 255.0, s=0.5)
ax1.set_title(f'DTU {SCENE} — Point Cloud ({len(xyz)} pts)')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# 카메라
ax2 = fig.add_subplot(122, projection='3d')
cam_positions = []
for img_id, img in imgs.items():
    R = qvec2rotmat(img.qvec)
    cam_pos = -R.T @ img.tvec
    cam_positions.append(cam_pos)
cam_positions = np.array(cam_positions)
ax2.scatter(cam_positions[:, 0], cam_positions[:, 1], cam_positions[:, 2], s=2, c='red', label='Cameras')
ax2.scatter(xyz[subsample, 0], xyz[subsample, 1], xyz[subsample, 2],
            c='gray', s=0.2, alpha=0.3, label='Points')
ax2.set_title(f'Cameras ({len(cam_positions)}) + Points')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n[DTU vs Mitsubishi 비교]")
print(f"  DTU 뷰 수: {len(imgs)} / Mitsubishi: 50 (전체 641 중 샘플링)")
print(f"  DTU 포인트 수: {len(xyz)} / Mitsubishi: GT depth 기반 back-projection")
cam_distances = np.linalg.norm(cam_positions, axis=1)
print(f"  DTU 카메라 반경: mean={cam_distances.mean():.3f}, std={cam_distances.std():.3f}")

## 4. Training

**논문 DTU 세팅(`scripts/dtu_eval.py:23`)**:
```
--depth_ratio 1.0 -r 2 --lambda_dist 1000
```

- `-r 2`: 해상도 1/2 (1600x1200 → 800x600)
- `--lambda_dist 1000`: depth distortion 정규화 강도 (미쓰비시 0.1의 10,000배! DTU는 정밀 geometry가 목적이므로 regularization 강하게 걸림)
- `--depth_ratio 1.0`: bounded scene → median depth
- `--eval` 없음: DTU 평가는 mesh Chamfer가 주 지표이므로 전체 뷰 학습에 사용
- `--lambda_normal`은 기본값 0.05 유지
- `--sh_degree`는 기본값 3 유지 (DTU는 실사 이미지, 조명 효과 포함)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT}

In [ ]:
# 학습 — scripts/dtu_eval.py:23 저자 하이퍼파라미터 그대로
# --depth_ratio 1.0 -r 2 --lambda_dist 1000
import subprocess
cmd = [
    "python", "train.py",
    "-s", DATA,
    "-m", OUTPUT,
    "--quiet",
    "--test_iterations", "-1",
    "--depth_ratio", "1.0",
    "-r", "2",
    "--lambda_dist", "1000",
    "--iterations", "30000",
    "--save_iterations", "7000", "30000",
    "--checkpoint_iterations", "15000", "30000",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


In [ ]:
# 체크포인트를 Google Drive에 백업
import shutil
os.makedirs(DRIVE_SAVE, exist_ok=True)
dst = os.path.join(DRIVE_SAVE, SCENE)
os.makedirs(dst, exist_ok=True)

for name in ["point_cloud", "cfg_args"]:
    src = os.path.join(OUTPUT, name)
    if os.path.exists(src):
        if os.path.isdir(src):
            target = os.path.join(dst, name)
            if os.path.exists(target):
                shutil.rmtree(target)
            shutil.copytree(src, target)
        else:
            shutil.copy2(src, dst)

import glob
for ckpt in glob.glob(os.path.join(OUTPUT, "chkpnt*.pth")):
    shutil.copy2(ckpt, dst)

print(f"Checkpoint saved to {dst}")

## 5. Mesh Extraction (핵심 평가 대상)

**논문 DTU 렌더링 세팅(`scripts/dtu_eval.py:32`)**:
```
--skip_train --depth_ratio 1.0 --num_cluster 1 --voxel_size 0.004 --sdf_trunc 0.016 --depth_trunc 3.0
```

- `--num_cluster 1`: 가장 큰 connected component 1개만 유지 (DTU는 단일 물체)
- `--voxel_size 0.004`: 4mm 해상도 (물체 크기 대비 충분히 세밀)
- `--sdf_trunc 0.016`: SDF 절단 거리 16mm (voxel_size × 4)
- `--depth_trunc 3.0`: 3m 너머 depth는 무시

In [ ]:
# 렌더링 + mesh — scripts/dtu_eval.py:32 저자 하이퍼파라미터 그대로
# --skip_train --depth_ratio 1.0 --num_cluster 1 --voxel_size 0.004 --sdf_trunc 0.016 --depth_trunc 3.0
import subprocess
cmd = [
    "python", "render.py",
    "-s", DATA,
    "-m", OUTPUT,
    "--iteration", "30000",
    "--quiet",
    "--skip_train",
    "--depth_ratio", "1.0",
    "--num_cluster", "1",
    "--voxel_size", "0.004",
    "--sdf_trunc", "0.016",
    "--depth_trunc", "3.0",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


In [ ]:
# 렌더링 결과 시각화 (test set이 비어있을 수 있으므로 train 먼저 체크)
import glob
from PIL import Image

for split in ["test", "train"]:
    dirs = sorted(glob.glob(os.path.join(OUTPUT, split, "ours_*")))
    if not dirs:
        continue
    latest = dirs[-1]
    render_dir = os.path.join(latest, "renders")
    gt_dir = os.path.join(latest, "gt")
    if not os.path.exists(render_dir):
        continue

    renders = sorted(glob.glob(os.path.join(render_dir, "*.png")))[:6]
    gts = sorted(glob.glob(os.path.join(gt_dir, "*.png")))[:6] if os.path.exists(gt_dir) else []

    rows = 2 if gts else 1
    fig, axes = plt.subplots(rows, len(renders), figsize=(3 * len(renders), 3 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    for i, r in enumerate(renders):
        if gts:
            axes[0, i].imshow(np.array(Image.open(gts[i]))); axes[0, i].set_title('GT', fontsize=8); axes[0, i].axis('off')
            axes[1, i].imshow(np.array(Image.open(r)));      axes[1, i].set_title('Rendered', fontsize=8); axes[1, i].axis('off')
        else:
            axes[0, i].imshow(np.array(Image.open(r))); axes[0, i].set_title('Rendered', fontsize=8); axes[0, i].axis('off')
    plt.suptitle(f'{SCENE} — {split}/{os.path.basename(latest)}')
    plt.tight_layout()
    plt.show()
    break

In [ ]:
# 메쉬 로드 & 통계
import trimesh

train_dirs = sorted(glob.glob(os.path.join(OUTPUT, "train", "ours_*")))
mesh_path = None
if train_dirs:
    for name in ["fuse_post.ply", "fuse.ply"]:
        cand = os.path.join(train_dirs[-1], name)
        if os.path.exists(cand):
            mesh_path = cand
            break

assert mesh_path, "메쉬 파일 없음 — render.py가 성공했는지 확인"
print(f"Mesh: {mesh_path}")

mesh = trimesh.load(mesh_path)
print(f"Vertices: {len(mesh.vertices)}")
print(f"Faces: {len(mesh.faces)}")
print(f"Bounding box: {mesh.bounds}")
print(f"Volume: {mesh.volume:.4f}" if mesh.is_volume else "Non-watertight")

# Drive 백업
shutil.copy2(mesh_path, os.path.join(DRIVE_SAVE, SCENE, os.path.basename(mesh_path)))
print(f"Saved to Drive.")

## 6. Chamfer Distance Evaluation (논문 Table 1 재현)

**DTU 공식 GT point cloud가 필요.**

공식 평가 스크립트: `scripts/eval_dtu/evaluate_single_scene.py`

필요한 파일 (DTU Official에서 받음):
```
{DTU_GT}/
├── Points/stl/stl{scan_id:03d}_total.ply   ← GT point cloud
├── ObsMask/ObsMask{scan_id}_10.mat         ← 평가 영역 마스크
└── SampleSet/MVS Data/...
```

In [ ]:
# DTU 공식 GT 구조 검증
# eval.py 가 요구하는 파일 (scan_id={SCAN_ID}):
#   {DTU_GT}/ObsMask/ObsMask{SCAN_ID}_10.mat   (eval.py:98)
#   {DTU_GT}/ObsMask/Plane{SCAN_ID}.mat        (eval.py:126)
#   {DTU_GT}/Points/stl/stl{SCAN_ID:03d}_total.ply  (eval.py:114, 반드시 3자리 zero-pad)

DRIVE_DTU_GT = "/content/drive/MyDrive/DTU_Official"

if os.path.exists(DRIVE_DTU_GT) and not os.path.exists(DTU_GT):
    shutil.copytree(DRIVE_DTU_GT, DTU_GT)
    print(f"Copied DTU_Official from Drive to {DTU_GT}")
elif os.path.exists(DTU_GT):
    print(f"DTU GT already at {DTU_GT}")
else:
    print(f"⚠️  {DRIVE_DTU_GT} 없음.")
    print("   수동 다운로드: https://roboimagedata.compute.dtu.dk/?page_id=36")
    print("   - SampleSet.zip (ObsMask*.mat, Plane*.mat 포함, ~5GB)")
    print("   - Points.zip (stl{N:03d}_total.ply)")

if os.path.exists(DTU_GT):
    expected = {
        "ObsMask": os.path.join(DTU_GT, "ObsMask", f"ObsMask{SCAN_ID}_10.mat"),
        "Plane":   os.path.join(DTU_GT, "ObsMask", f"Plane{SCAN_ID}.mat"),
        "STL_pc":  os.path.join(DTU_GT, "Points", "stl", f"stl{SCAN_ID:03d}_total.ply"),
    }
    print(f"\n=== DTU_Official / scan{SCAN_ID} 필수 파일 확인 ===")
    all_ok = True
    for k, p in expected.items():
        ok = os.path.exists(p)
        print(f"  {'✅' if ok else '❌'} {k}: {p}")
        all_ok = all_ok and ok
    if not all_ok:
        print("\n⚠️  GT 파일 누락.")
        print(f"   SampleSet.zip → SampleSet/MVS Data/ObsMask/*.mat 를 {DTU_GT}/ObsMask/ 로")
        print(f"   Points.zip → Points/stl/stl*.ply 를 {DTU_GT}/Points/stl/ 로")

In [ ]:
# 공식 평가 스크립트 실행 (저자 scripts/dtu_eval.py:45-50 와 동일 방식)
#
# 주의사항:
# 1. evaluate_single_scene.py:16 `import render_utils as rend_util` → cwd가 scripts/eval_dtu/ 여야 함
# 2. 저자는 --output_dir을 {script_dir}/tmp/scan{id} 로 두지만, 임의 경로도 OK
# 3. --mask_dir 은 scan 상위 폴더 (evaluate_single_scene.py:125 에서 /scan{id} 붙임)
# 4. eval.py:158 에서 "mean_d2s mean_s2d overall" 을 stdout에 찍고, results.json도 저장
import subprocess

EVAL_DIR = os.path.join(REPO, "scripts", "eval_dtu")
EVAL_OUT = os.path.join(EVAL_DIR, "tmp", f"scan{SCAN_ID}")
os.makedirs(EVAL_OUT, exist_ok=True)

if os.path.exists(os.path.join(EVAL_DIR, "evaluate_single_scene.py")) and os.path.exists(DTU_GT):
    cmd = [
        "python", "evaluate_single_scene.py",
        "--input_mesh", mesh_path,
        "--scan_id", str(SCAN_ID),
        "--output_dir", EVAL_OUT,
        "--mask_dir", DTU_ROOT,
        "--DTU", DTU_GT,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=EVAL_DIR, check=True)
else:
    print(f"eval script exists: {os.path.exists(os.path.join(EVAL_DIR, 'evaluate_single_scene.py'))}")
    print(f"DTU_GT exists: {os.path.exists(DTU_GT)}")
    print("평가 건너뜀.")


In [ ]:
# 결과 파싱 — eval.py:160-166 에서 results.json 에 저장됨
import json as _json

result_file = os.path.join(EVAL_OUT, "results.json")
if os.path.exists(result_file):
    with open(result_file) as f:
        results = _json.load(f)
    print(f"=== DTU {SCENE} Chamfer Evaluation ===")
    print(_json.dumps(results, indent=2))

    # 키 예시: mean_d2s (accuracy, mm), mean_s2d (completeness, mm), overall_chamfer (mm)
    overall = results.get('overall', results.get('overall_chamfer'))
    if overall is not None:
        print(f"\n→ Overall Chamfer: {overall:.4f} mm")

    print("\n[논문 기준값 — README Table, Chamfer mm (lower is better)]")
    ref_table = {
        24: 0.48, 37: 0.91, 40: 0.39, 55: 0.39, 63: 1.01,
        65: 0.83, 69: 0.81, 83: 1.36, 97: 1.27, 105: 0.76,
        106: 0.70, 110: 1.40, 114: 0.40, 118: 0.76, 122: 0.52,
    }
    if SCAN_ID in ref_table:
        print(f"  scan{SCAN_ID}: 논문 {ref_table[SCAN_ID]} mm")
    print(f"  전체 15 scan 평균: 0.80 mm (논문), 0.74 mm (저자 재현)")
else:
    print(f"results.json not found at {result_file}")
    # Drive 백업
    pass

# Drive 백업
if os.path.exists(EVAL_OUT):
    shutil.copytree(EVAL_OUT, os.path.join(DRIVE_SAVE, SCENE, "eval"), dirs_exist_ok=True)
    print(f"\nEval results backed up to Drive.")

## 7. 전체 15 Scan 배치 실행 (선택)

`scripts/dtu_eval.py`를 그대로 호출해서 15개 전부 학습+평가.
**Colab에서는 권장하지 않음** — T4 기준 scan당 ~20분, 15개 × (학습 + 렌더 + 평가) ≈ 6~8시간.

In [ ]:
# 전체 15 scan 배치 — 저자 scripts/dtu_eval.py 그대로 호출
# 주의: 이 스크립트 line 35에 "-m<path>" 타이포 있지만 argparse가 관대해서 작동하긴 함.
# {ROOT}/eval/dtu_all/scan{id}/ 구조로 결과가 쌓임.
#
# T4 기준 scan당 학습 ~25분 + 렌더 ~3분 + 평가 ~1분 → 15개 전부 ≈ 7~8시간
# 주석 해제 시 실행

# !cd {REPO} && python scripts/dtu_eval.py \
#     --dtu {DTU_ROOT} \
#     --DTU_Official {DTU_GT} \
#     --output_path {ROOT}/eval/dtu_all

# 학습/렌더만 스킵하고 평가만 돌리려면 (이미 mesh가 있는 경우):
# !cd {REPO} && python scripts/dtu_eval.py \
#     --skip_training --skip_rendering \
#     --dtu {DTU_ROOT} \
#     --DTU_Official {DTU_GT} \
#     --output_path {ROOT}/eval/dtu_all

print("주석 해제 시 논문 Table 전체 재현.")
print("참고: dtu_eval.py:35 의 '-m<path>' 는 타이포지만 argparse 파싱은 통과함 (저자 원본 유지).")

## 8. Mitsubishi vs DTU 결과 비교표

두 파이프라인을 돌린 뒤 이 셀에서 비교표를 생성.
`pipeline_colab.ipynb`(미쓰비시) 실행 결과가 `/content/drive/MyDrive/2dgs_results`에 있어야 함.

In [ ]:
import json

MITSUBISHI_DIR = "/content/drive/MyDrive/2dgs_results"
DTU_DIR = os.path.join(DRIVE_SAVE, SCENE)

summary = {
    "Mitsubishi": {
        "data_type": "Blender synthetic",
        "num_views": 50,
        "resolution": "800x800",
        "texture": "monochrome gray (edge 0.54%)",
        "lambda_dist": 0.1,
        "lambda_normal": 0.05,
        "sh_degree": 1,
        "depth_ratio": 1.0,
        "eval_metric": "PSNR/SSIM + depth MAE vs GT",
    },
    f"DTU_{SCENE}": {
        "data_type": "Real photos (NeuS preprocessed)",
        "num_views": len(imgs) if 'imgs' in dir() else "~49-64",
        "resolution": "1600x1200 (-r 2 → 800x600)",
        "texture": "rich (실사 이미지)",
        "lambda_dist": 1000,
        "lambda_normal": 0.05,
        "sh_degree": 3,
        "depth_ratio": 1.0,
        "eval_metric": "Chamfer distance (mm) vs GT point cloud",
    },
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\n[핵심 관찰 포인트]")
print("1. lambda_dist가 10,000배 차이 — DTU는 실측 스케일 + 정밀 geometry 요구, "
      "Mitsubishi는 합성 씬이라 약한 regularization으로 충분")
print("2. 평가지표가 다름 — Mitsubishi는 novel-view PSNR이 주, DTU는 mesh Chamfer가 주")
print("3. 텍스처 풍부도 차이 → Mitsubishi에서 normal consistency loss의 의존도가 훨씬 높음")
print("4. 뷰 수 차이 — Mitsubishi는 64°/뷰로 조밀, DTU는 턴테이블 기반 분산")